# Advanced 04 — Independence, Borel-Cantelli, and Zero-One Laws

Practice notebook for the **"Independence, Borel-Cantelli, and Zero-One Laws"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Independence of sigma-algebras and random variables

Sigma-algebras \(\mathcal{F}_1,\dots,\mathcal{F}_n\) are **independent** if

$$
P(A_1\cap\cdots\cap A_n) \ =\ \prod_{i=1}^{n} P(A_i)
\qquad\text{for all } A_i\in\mathcal{F}_i.
$$

Random variables \(X_i\) are independent when the sigma-algebras
\(\sigma(X_i)\) they generate are independent. Pairwise independence is
*not* enough — we need the product rule for *every* subfamily. Below we
construct an explicit probability space with two genuinely independent
events \(A,B\) and verify \(P(A\cap B)=P(A)P(B)\) both analytically and by
Monte Carlo. We also exhibit a classic three-event example that is
pairwise independent but **not** mutually independent, to flag the
distinction.


In [ ]:
# Two independent events on the product space Omega = {0,1} x {0,1}.
# A = "first coordinate is 1", B = "second coordinate is 1".
# Under the uniform product measure, P(A)=P(B)=1/2 and P(A cap B)=1/4.

rng = np.random.default_rng(42)

# Analytical probabilities on the uniform product space.
P_A = 0.5
P_B = 0.5
P_AandB = 0.25
print("Analytical:")
print(f"  P(A)      = {P_A:.4f}")
print(f"  P(B)      = {P_B:.4f}")
print(f"  P(A cap B)= {P_AandB:.4f}")
print(f"  P(A)P(B)  = {P_A*P_B:.4f}")
print(f"  independent? {abs(P_AandB - P_A*P_B) < 1e-12}")

# Monte Carlo: sample (X,Y) with X,Y ~ Bernoulli(1/2) independently.
N = 200_000
X = rng.integers(0, 2, size=N)
Y = rng.integers(0, 2, size=N)
A_mask = X == 1
B_mask = Y == 1
emp_A = A_mask.mean()
emp_B = B_mask.mean()
emp_AB = (A_mask & B_mask).mean()
print()
print(f"Monte Carlo (N={N:,}):")
print(f"  P_hat(A)      = {emp_A:.4f}")
print(f"  P_hat(B)      = {emp_B:.4f}")
print(f"  P_hat(A cap B)= {emp_AB:.4f}")
print(f"  P_hat(A)P_hat(B) = {emp_A*emp_B:.4f}")
print(f"  |P_hat(A cap B) - P_hat(A)P_hat(B)| = {abs(emp_AB - emp_A*emp_B):.4f}")

# Classic counterexample: pairwise independent but NOT mutually independent.
# Toss two fair coins: X,Y in {0,1}^2 uniform. Let
#   A = {X=1}, B = {Y=1}, C = {X xor Y = 1}  (i.e. C = X xor Y).
# Then P(A)=P(B)=P(C)=1/2, each pair is independent, but P(A cap B cap C)=0
# != P(A)P(B)P(C) = 1/8, so (A,B,C) are NOT mutually independent.
Xv = rng.integers(0, 2, size=N)
Yv = rng.integers(0, 2, size=N)
Av = Xv == 1
Bv = Yv == 1
Cv = (Xv ^ Yv) == 1
print()
print("Pairwise-but-not-mutual counterexample:")
print(f"  P(A)={Av.mean():.3f}  P(B)={Bv.mean():.3f}  P(C)={Cv.mean():.3f}")
print(f"  P(A cap B)={ (Av&Bv).mean():.3f}  P(A)P(B)={Av.mean()*Bv.mean():.3f}")
print(f"  P(A cap C)={ (Av&Cv).mean():.3f}  P(A)P(C)={Av.mean()*Cv.mean():.3f}")
print(f"  P(B cap C)={ (Bv&Cv).mean():.3f}  P(B)P(C)={Bv.mean()*Cv.mean():.3f}")
print(f"  P(A cap B cap C)={ (Av&Bv&Cv).mean():.4f}  P(A)P(B)P(C)={Av.mean()*Bv.mean()*Cv.mean():.4f}")
print("  -> pairs look independent, but the triple is NOT mutually independent.")


**Your turn:** Construct three events \(A,B,C\) on a suitably rich
probability space that *are* mutually independent (i.e.
\(P(A\cap B\cap C)=P(A)P(B)P(C)\) and similarly for every subfamily), and
verify it numerically. How many samples do you need before the empirical
triple intersection is within \(10^{-3}\) of the product of marginals?


---
## Part 2 — First Borel-Cantelli lemma

**Theorem (first Borel-Cantelli).** If \(A_1,A_2,\dots\) are events with

$$
\sum_{n=1}^{\infty} P(A_n) \ <\ \infty,
\qquad\text{then}\qquad
P(A_n\ \text{i.o.}) = 0,
$$

where "i.o." means "infinitely often", i.e.
\(\{A_n\ \text{i.o.}\}=\limsup_{n} A_n=\bigcap_{m}\bigcup_{n\geq m} A_n\).

The intuition: if the *total expected count*
\(\sum_n P(A_n)\) is finite, then almost surely only finitely many
\(A_n\) occur. Crucially, this lemma needs **no independence assumption**.

We simulate with \(P(A_n)=1/n^{2}\), for which
\(\sum_{n\geq 1}1/n^{2}=\pi^{2}/6<\infty\). Across many independent
realizations of the whole sequence, the total number of occurrences should
be finite (and small) with probability 1.


In [ ]:
# First Borel-Cantelli: P(A_n) = 1/n^2 is summable -> only finitely many occur.
rng = np.random.default_rng(42)

n_max = 5_000                     # truncate the infinite sequence
n = np.arange(1, n_max + 1)
p_n = 1.0 / n**2                  # P(A_n) = 1/n^2

print(f"Sum of P(A_n) for n=1..{n_max}: {p_n.sum():.4f}")
print(f"Theoretical infinite sum  pi^2/6 = {np.pi**2/6:.4f}")

# Simulate R independent realizations of the whole sequence {A_n}.
R = 20_000                        # number of full-sequence replicates
# Each row is one realization; entry (r, n) = 1 if A_n occurred.
occurrences = rng.random(size=(R, n_max)) < p_n[None, :]
total_counts = occurrences.sum(axis=1)   # total number of A_n that occurred per replicate

print()
print(f"Replicates: {R:,}")
print(f"Mean number of occurrences per replicate: {total_counts.mean():.3f}")
print(f"  (expected count = sum P(A_n) = {p_n.sum():.3f})")
print(f"Max occurrences observed in any replicate: {total_counts.max()}")
print(f"Fraction of replicates with >= 20 occurrences: {(total_counts>=20).mean():.4f}")
print(f"Fraction with >= 100 occurrences:            {(total_counts>=100).mean():.6f}")

# P(A_n i.o.) = 0 means: with probability 1 the total count is finite.
# Empirically the count distribution is tight around the finite mean sum.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(total_counts, bins=np.arange(-0.5, total_counts.max()+1.5, 1),
             color="steelblue", edgecolor="white", density=True)
axes[0].axvline(p_n.sum(), color="crimson", linestyle="--",
               label=rf"$\sum P(A_n)={p_n.sum():.2f}$")
axes[0].set_xlabel("number of $A_n$ that occurred")
axes[0].set_ylabel("frequency")
axes[0].set_title("First Borel-Cantelli: finite total count")
axes[0].legend()

# Running count of occurrences for a few replicates: it plateaus (finite).
for r in range(8):
    running = np.cumsum(occurrences[r])
    axes[1].plot(n, running, alpha=0.7)
axes[1].set_xlabel("n")
axes[1].set_ylabel(r"running count $\sum_{k\leq n} 1_{A_k}$")
axes[1].set_title("Running count plateaus -> only finitely many $A_n$ occur")
axes[1].set_ylim(bottom=0)
plt.tight_layout(); plt.show()

print("Conclusion: with probability 1, only finitely many A_n occur, as BC1 promises.")


---
## Part 3 — Second Borel-Cantelli lemma

**Theorem (second Borel-Cantelli).** If \(A_1,A_2,\dots\) are **independent**
events with

$$
\sum_{n=1}^{\infty} P(A_n) = \infty,
\qquad\text{then}\qquad
P(A_n\ \text{i.o.}) = 1.
$$

Here independence is essential. The "borderline" case
\(P(A_n)=1/n\) has \(\sum 1/n=\infty\) but the \(A_n\) we simulate
below are constructed to be independent, so the lemma applies and the events
occur infinitely often. To make the divergence unambiguous, we also use a
constant \(p_n=c\in(0,1]\), for which \(\sum_n c=\infty\) trivially and
the i.o. conclusion is obvious. We verify **mutual independence** of the
constructed events numerically by checking
\(P(A_i\cap A_j)=P(A_i)P(A_j)\) on pairs, then count how often the
sequence fires across a long run.


In [ ]:
# Second Borel-Cantelli: independent events with non-summable probabilities.
rng = np.random.default_rng(42)

# (a) Constant p_n = c in (0,1]: sum diverges, events independent -> i.o. with prob 1.
c = 0.05
n_max = 5_000
n = np.arange(1, n_max + 1)

# Construct independent A_n with P(A_n) = c via iid uniforms.
U = rng.random(size=(n_max,))
A_const = U < c                      # independent Bernoulli(c) indicators

print(f"Constant p_n = c = {c}")
print(f"Sum of p_n over n=1..{n_max}: {n_max * c:.1f}  (diverges as n_max -> inf)")
print(f"Empirical P(A_n) = {A_const.mean():.4f}")

# Verify mutual independence numerically on a block of pairs.
# Check P(A_i cap A_j) = P(A_i) P(A_j) for many (i,j) pairs.
K = 200                              # use first K events for pairwise independence check
A_block = U[:K] < c
# All pairs (i<j) within the block:
cnt_pairs = 0
max_err = 0.0
for i in range(K):
    Ai = U[i] < c
    for j in range(i+1, K):
        Aj = U[j] < c
        p_joint = float(Ai and Aj)               # single realization -> 0/1
        # Aggregate over replicates instead: re-draw independent copies.
        cnt_pairs += 1
# Better: estimate pairwise independence by Monte Carlo with replicates.
R = 50_000
Ai_draws = rng.random(size=R) < c
Aj_draws = rng.random(size=R) < c
p_A = Ai_draws.mean()
p_B = Aj_draws.mean()
p_AB = (Ai_draws & Aj_draws).mean()
print()
print("Mutual-independence check (two independent Bernoulli(c) copies):")
print(f"  P_hat(A)   = {p_A:.4f}")
print(f"  P_hat(B)   = {p_B:.4f}")
print(f"  P_hat(A&B) = {p_AB:.4f}")
print(f"  P_hat(A)P_hat(B) = {p_A*p_B:.4f}")
print(f"  |P(A&B) - P(A)P(B)| = {abs(p_AB - p_A*p_B):.4f}  -> independent")

# Running count of occurrences: grows roughly like c * n (linear -> infinity).
running_const = np.cumsum(A_const.astype(int))
print()
print(f"Number of A_n that occurred by n={n_max}: {running_const[-1]}")
print(f"  (expected c * n_max = {c*n_max:.1f})")
print("Since the running count diverges, A_n occurs infinitely often.")

# (b) Borderline case P(A_n) = 1/n: sum diverges (harmonic), independent -> i.o.
p_n_harm = 1.0 / n
U2 = rng.random(size=(n_max,))
A_harm = U2 < p_n_harm                # independent Bernoulli(1/n)
running_harm = np.cumsum(A_harm.astype(int))
print()
print(f"P(A_n)=1/n: sum_{'{n<=N}'} 1/n = {np.sum(1.0/n):.2f}  (-> inf)")
print(f"Number of A_n with P=1/n that occurred by n={n_max}: {running_harm[-1]}")
print("  harmonic series diverges -> with independence, A_n occurs i.o.")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(n, running_const, label=rf"$p_n=c={c}$: $\sim c\,n$ (linear, -> inf)", color="steelblue")
ax.plot(n, running_harm, label=r"$p_n=1/n$: $\sim \log n$ (-> inf)", color="crimson")
ax.set_xlabel("n"); ax.set_ylabel(r"running count $\sum_{k\leq n} 1_{A_k}$")
ax.set_title("Second Borel-Cantelli: non-summable + independent => i.o.")
ax.legend(); plt.tight_layout(); plt.show()


**Your turn:** Pick \(p_n=1/(n\log n)\) for \(n\geq 2\). Does
\(\sum p_n\) converge or diverge? Simulate independent \(A_n\) with these
probabilities and report whether the running count diverges — does your
numerical answer agree with the second Borel-Cantelli lemma?


---
## Part 4 — Tail sigma-algebra and Kolmogorov 0-1 law

For a sequence of random variables \((X_n)\), the **tail sigma-algebra** is

$$
\mathcal{T} \ =\ \bigcap_{n=1}^{\infty} \sigma(X_k : k \geq n).
$$

**Kolmogorov's zero-one law.** If the \(X_n\) are independent, then every
\(A\in\mathcal{T}\) has \(P(A)\in\{0,1\}\). Tail events depend only on
what happens "in the long run"; there is no intermediate randomness at
infinity.

A canonical tail event is
\(\{\limsup_{n} X_n = +\infty\}\), or, for events \(A_n\), the limsup
\(\limsup_n A_n=\{A_n\ \text{i.o.}\}\). When the \(A_n\) are generated
from independent \(X_n\) (e.g. \(A_n=\{X_n\in B\}\) for a fixed Borel
\(B\)), \(\{A_n\ \text{i.o.}\}\) is a tail event, so its probability is
\(0\) or \(1\). Below we estimate the probability of the limsup event for
two different \(p_n\) schedules and confirm it is essentially 0 (BC1 regime)
or essentially 1 (BC2 regime) — never anything in between.


In [ ]:
# Kolmogorov 0-1 law: P(limsup A_n) is 0 or 1 for independent A_n.
rng = np.random.default_rng(42)

# The limsup event {A_n i.o.} = union_m intersection_{N>=m} ... no, it is
#   limsup A_n = intersect_m union_{n>=m} A_n.
# We approximate it over a long horizon n_max: a replicate "sees i.o." if its
# running count is still increasing at the end, i.e. infinitely many hits.
# For a finite-horizon proxy we flag a replicate as "i.o." if the count of
# hits in the last 25% of the sequence is > 0 (i.e. hits keep occurring far out).

n_max = 4_000
n = np.arange(1, n_max + 1)
R = 5_000                            # replicates
quarter = n_max // 4
start_last = n_max - quarter          # start of the "last quarter" window

def estimate_limsup(p_n, label):
    # Each row: one independent realization of the sequence {A_n}.
    draws = rng.random(size=(R, n_max))
    occ = draws < p_n[None, :]
    total = occ.sum(axis=1)
    last_q = occ[:, start_last:].sum(axis=1)
    # "i.o." proxy: hits keep appearing in the tail window.
    io_mask = last_q > 0
    p_io = io_mask.mean()
    print(f"{label}:")
    print(f"  sum P(A_n) (n<= {n_max}) = {p_n.sum():.3f}")
    print(f"  mean total hits per replicate       = {total.mean():.2f}")
    print(f"  P_hat(A_n i.o.) proxy               = {p_io:.4f}")
    print(f"  -> essentially {'1 (BC2)' if p_io > 0.5 else '0 (BC1)'}")
    return p_io, total

print("=" * 60)
# BC1 regime: summable -> P(limsup) = 0.
p1 = 1.0 / n**2
p_io1, total1 = estimate_limsup(p1, "BC1: P(A_n)=1/n^2 (summable)")

print("-" * 60)
# BC2 regime: non-summable, independent -> P(limsup) = 1.
p2 = np.full(n_max, 0.05)
p_io2, total2 = estimate_limsup(p2, "BC2: P(A_n)=0.05 constant (non-summable)")

print("=" * 60)
print("In both regimes the empirical P(limsup A_n) is essentially 0 or 1,")
print("never ~0.5: this is the Kolmogorov 0-1 law in action.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].hist(total1, bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(p1.sum(), color="crimson", linestyle="--", label=rf"$\sum P(A_n)={p1.sum():.1f}$")
axes[0].set_title(rf"BC1: $P(A_n\text{{ i.o.}})\approx {p_io1:.3f}$")
axes[0].set_xlabel("total hits per replicate"); axes[0].legend()
axes[1].hist(total2, bins=40, color="crimson", edgecolor="white")
axes[1].axvline(p2.sum(), color="steelblue", linestyle="--", label=rf"$\sum P(A_n)={p2.sum():.0f}$")
axes[1].set_title(rf"BC2: $P(A_n\text{{ i.o.}})\approx {p_io2:.3f}$")
axes[1].set_xlabel("total hits per replicate"); axes[1].legend()
plt.tight_layout(); plt.show()


**Your turn:** Consider the tail event
\(\{\limsup_n X_n = +\infty\}\) for \(X_n\sim\text{Bernoulli}(p)\) iid.
Without simulation, argue using the second Borel-Cantelli lemma why
\(P(\limsup X_n = +\infty)\) must be \(1\) (think
\(A_n=\{X_n\geq 1/2\}\)). Then estimate the event numerically and check it
is effectively \(1\).


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. State the first and second Borel-Cantelli lemmas precisely, and
identify exactly where the **independence** hypothesis enters the second
lemma (and is absent from the first). Give a one-sentence example showing the
second lemma can fail without independence.

2. Let \(A_n\) be independent events with \(P(A_n)=1/n^{2}\). Compute
\(\sum_{n=1}^{\infty} P(A_n)\) and state whether \(P(A_n\ \text{i.o.})\)
is 0 or 1. Justify with the appropriate lemma.

3. Construct independent events \(A_n\) with \(P(A_n)=1/n\) for
\(n\geq 1\). Show \(\sum_n P(A_n)=\infty\) and conclude
\(P(A_n\ \text{i.o.})=1\). Verify mutual independence numerically by
checking \(P(A_i\cap A_j)=P(A_i)P(A_j)\) on simulated pairs with a
reproducible seed.

4. Define the tail sigma-algebra \(\mathcal{T}\) for a sequence
\((X_n)\). State the Kolmogorov 0-1 law and give an example of a tail event
whose probability is forced to be \(0\) or \(1\). Why is the event
\(\{X_1=0\}\) *not* a tail event?

5. Give an example of events \(A_n\) (not independent) for which
\(\sum_n P(A_n)=\infty\) but \(P(A_n\ \text{i.o.})=0\). This shows the
independence hypothesis in the second Borel-Cantelli lemma cannot simply be
dropped.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** **First Borel-Cantelli:** if \(\sum_n P(A_n)<\infty\)
then \(P(A_n\ \text{i.o.})=0\) — no independence needed. **Second
Borel-Cantelli:** if the \(A_n\) are **independent** and
\(\sum_n P(A_n)=\infty\), then \(P(A_n\ \text{i.o.})=1\). Independence
enters the second lemma through the Borel-Cantelli lower bound:
\(P(\bigcup_{n=m}^{N}A_n)=1-\prod_{n=m}^{N}(1-P(A_n))\), which uses
independence to factor the complement; letting \(N\to\infty\) and using
\(\sum P(A_n)=\infty\Rightarrow\prod(1-P(A_n))=0\) gives
\(P(\bigcup_{n\geq m}A_n)=1\) for every \(m\), hence the limsup has
probability 1. Without independence the factorization fails: e.g. take
\(A_n=A\) for every \(n\) with \(0<P(A)<1\) — then
\(\sum_n P(A_n)=\infty\) but \(A_n\ \text{i.o.}=A\), whose probability is
\(P(A)\in(0,1)\), not 1.

**2.** \(\sum_{n=1}^{\infty}1/n^{2}=\pi^{2}/6<\infty\). By the **first**
Borel-Cantelli lemma (which needs no independence), \(P(A_n\ \text{i.o.})=0\).
The independence of the \(A_n\) is irrelevant here — summability alone
settles it.

**3.** \(\sum_{n=1}^{N}1/n\sim\log N\to\infty\). Since the \(A_n\) are
independent, the **second** Borel-Cantelli lemma gives
\(P(A_n\ \text{i.o.})=1\). Numerical check (seed 42):

```python
import numpy as np
rng = np.random.default_rng(42)
R = 100_000
Ai = rng.random(R) < 1/3     # P(A_i) = 1/3
Aj = rng.random(R) < 1/7     # P(A_j) = 1/7
print(abs((Ai & Aj).mean() - Ai.mean()*Aj.mean()))   # ~1e-3 -> independent
```
The empirical \(P(A_i\cap A_j)\) matches \(P(A_i)P(A_j)\), confirming
mutual independence on pairs.

**4.** \(\mathcal{T}=\bigcap_{n=1}^{\infty}\sigma(X_k:k\geq n)\). A tail
event is one whose occurrence is unchanged by changing *finitely* many of the
\(X_n\). The Kolmogorov 0-1 law: if the \(X_n\) are independent then every
\(A\in\mathcal{T}\) satisfies \(P(A)\in\{0,1\}\). Example:
\(\{\limsup_n X_n=+\infty\}\) is a tail event (changing finitely many
\(X_n\) does not affect a limsup), so its probability is 0 or 1. The event
\(\{X_1=0\}\) is **not** a tail event because dropping the
\(\sigma\)-algebra term \(\sigma(X_1)\) (i.e. taking \(n\geq 2\)) removes
all information about \(X_1\); \(\{X_1=0\}\notin\sigma(X_k:k\geq n)\) for
any \(n\geq 2\), so it is not in \(\mathcal{T}\).

**5.** Let \(A_n=A\) for all \(n\), with \(0<P(A)<1\) (take
\(A=\{X_1=0\}\) for \(X_1\sim\text{Bernoulli}(1/2)\)). Then
\(\sum_n P(A_n)=\sum_n P(A)=\infty\), but
\(\{A_n\ \text{i.o.}\}=A\) (if \(A\) happens once, it happens at every
\(n\); if it never happens, it happens zero times), so
\(P(A_n\ \text{i.o.})=P(A)\in(0,1)\). The \(A_n\) are perfectly
correlated, not independent, and the second Borel-Cantelli conclusion fails —
exactly because independence was used to factor the product
\(\prod_n(1-P(A_n))\).


</details>
